# Planewave DFT with Abinit and AbiPy -- Part 2: Analysis and plotting

**CEMDI 2026 -- hands-on session**

This is the second of two notebooks that make up the companion material for
the 3-hour CEMDI tutorial on planewave density-functional theory (DFT) with
[Abinit](https://www.abinit.org) and [AbiPy](https://github.com/abinit/abipy).
Part 1, [`1_running_calculations.ipynb`](1_running_calculations.ipynb),
built and launched every `Flow` we look at here; this notebook assumes they
have already run to completion (they have -- everything below reads from
`flow_*/` folders that ship, pre-run, next to this notebook).

This notebook is about *analysis and plotting*: opening the native netcdf
output files (`GSR.nc`, `DDB`, ...) with `abilab.abiopen` and plotting what
they contain. AbiPy `Robot`s -- which collect *several* output files of the
same kind into a single pandas `DataFrame` -- and the command-line tools
(`abirun.py`, `abiopen.py`, `abicomp.py`) are only introduced in the last
section, once you've seen what a single output file looks like on its own.

**Outline**

1. Setup
2. Ground-state total energy of GaAs
3. Band structures: GaAs vs Si
4. Phonons from DFPT
5. Robots and the command-line tools


## 1. Setup

Same as Part 1 -- this is a separate notebook (and kernel), so we need our
own imports:


In [ ]:
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings for the tutorial

import numpy as np

from abipy import abilab
abilab.enable_notebook()  # Tell AbiPy we are running inside a notebook

# Show matplotlib figures embedded in the notebook.
%matplotlib inline

import workshop_lib as wlib

`workshop_lib` is the same module used in Part 1 to build every flow
analyzed below -- here we only need it once, for `gaas_structure()` (to
get e.g. the atom count), in Section 5.1.


## 2. Ground-state total energy of GaAs

Results from the flow built in Part 1, Section 3 (`flow_gaas_gstate/`).
First, a quick sanity check on the main output file:


In [ ]:
abo = abilab.abiopen("flow_gaas_gstate/w0/t0/run.abo")
print(abo)

In [ ]:
print(abo.events)  # Warnings / Comments / Errors, if any

In [ ]:
abo.plot();  # SCF cycle: total energy / residuals vs iteration

The `GSR.nc` file produced by the SCF run has the total energy, forces,
stress tensor and the Kohn-Sham band structure at the end of the run:

In [ ]:
with abilab.abiopen("flow_gaas_gstate/w0/t0/outdata/out_GSR.nc") as gsr:
    print("energy:", gsr.energy, "eV")
    print("pressure:", gsr.pressure, "GPa")
    print(gsr.energy_terms)

> **Warning.** `abiopen` keeps a file descriptor open. Either call
> `.close()` explicitly, or -- better -- use it as a context manager
> (`with abilab.abiopen(...) as f:`) as we just did, so that the file is
> closed automatically.

## 3. Band structures: GaAs vs Si

Results from the flows built in Part 1, Section 5 (`flow_gaas_ebands/`,
`flow_si_ebands/`): a ground-state run on a homogeneous k-mesh followed by
a non-self-consistent (NSCF) run along a high-symmetry k-path.


In [ ]:
with abilab.abiopen("flow_gaas_ebands/w0/t1/outdata/out_GSR.nc") as gsr:
    gaas_ebands = gsr.ebands

fig = gaas_ebands.plot(color="b", show=False)
fig.gca().set_ylim(-10, 10)
fig.gca().set_title("GaAs");

In [ ]:
with abilab.abiopen("flow_si_ebands/w0/t1/outdata/out_GSR.nc") as gsr:
    si_ebands = gsr.ebands

fig = si_ebands.plot(color="g", show=False)
fig.gca().set_ylim(-10, 10)
fig.gca().set_title("Si");

GaAs has a direct gap at $\Gamma$, while silicon's fundamental gap is
indirect ($\Gamma \to$ near X). Both are visible on the two plots above --
this is a good moment to discuss why LDA/GGA systematically underestimate
these gaps, and what the standard corrections are ($GW$, hybrid
functionals, scissor operators).

> **Note.** `ElectronBands` objects also expose `.plot_with_edos(edos)` to
> overlay the density of states next to the bands, and `to_bxsf()` /
> `.plot_fermi_surface()` for metals -- not needed for GaAs/Si, but worth
> knowing about.

## 4. Phonons from DFPT

Results from the flow built in Part 1, Section 7 (`flow_gaas_phonons/`).
All the DFPT results (dynamical matrices, Born effective charges,
dielectric tensor) are merged into a single `DDB` file -- the entry point
for essentially all the post-processing below:


In [ ]:
ddb = abilab.abiopen("flow_gaas_phonons/w1/outdata/out_DDB")
print(ddb)

`ddb.anaget_phbst_and_phdos_files` calls `anaddb` behind the scenes to
Fourier-interpolate the dynamical matrix onto a dense q-mesh (for the
phonon DOS) and along a high-symmetry q-path (for the phonon band
structure):

In [ ]:
phbst_file, phdos_file = ddb.anaget_phbst_and_phdos_files(
    nqsmall=10, ndivsm=10, asr=2, chneut=1, dipdip=1,
    dos_method="tetra", lo_to_splitting=True)

phbands = phbst_file.phbands
phbands.plot_with_phdos(phdos_file.phdos);

> **Note.** `lo_to_splitting=True` accounts for the LO-TO splitting at
> $\Gamma$ that polar materials like GaAs show, using the Born effective
> charges and the dielectric tensor also stored in the `DDB`.

Born effective charges and the dielectric tensor can be inspected directly:

In [ ]:
epsinf, becs = ddb.anaget_epsinf_and_becs()
print("Electronic dielectric tensor:\n", epsinf)
print("\nBorn effective charges:")
print(becs)

In [ ]:
phbst_file.close()
phdos_file.close()
ddb.close()

## 5. Robots and the command-line tools

So far we've analyzed one output file at a time with `abilab.abiopen`. Two
Abinit parameters control the accuracy of a planewave DFT calculation --
the plane-wave cutoff `ecut` and the k-point mesh density -- and comparing
many runs that only differ by one such parameter is exactly what AbiPy
`Robot`s (`GsrRobot`, `DdbRobot`, ...) are for: they collect many files of
one kind into a single pandas `DataFrame` you can filter, sort and plot.
`Robot`s are the programmatic equivalent of the `abicomp.py` command-line
tool -- more on the command-line tools at the end of this section.


### 5.1 ecut convergence

Results from the flow built in Part 1, Section 4.1 (`flow_gaas_convecut/`).
A `GsrRobot` collects every `GSR.nc` file into a single `DataFrame`:


In [ ]:
with abilab.GsrRobot.from_dir("flow_gaas_convecut") as robot:
    table = robot.get_dataframe()

table[["ecut", "energy", "pressure"]].sort_values("ecut")

`abipy.tools.plotting.ConvergenceAnalyzer` fits/plots a quantity against a
convergence parameter and reports the smallest parameter value for which
the target quantity stays within a given tolerance -- this is exactly what
`Analysis/005-GaAs-conv-ecut/plot_ecut_conv.py` does:

In [ ]:
from abipy.tools.plotting import ConvergenceAnalyzer

ecut_Ha = table.sort_values("ecut")["ecut"].tolist()
ene_per_atom_eV = (table.sort_values("ecut")["energy"] / len(wlib.gaas_structure())).tolist()

ca = ConvergenceAnalyzer.from_xy_label_vals(
    "ecut (Ha)", ecut_Ha, "E/natom (eV)", ene_per_atom_eV, tols=1e-3)
ca.plot();

### 5.2 k-point convergence

Results from the flow built in Part 1, Section 4.2 (`flow_gaas_convkpt/`).


In [ ]:
with abilab.GsrRobot.from_dir("flow_gaas_convkpt") as robot:
    table = robot.get_dataframe()

table[["energy", "pressure"]]

`Analysis/006-GaAs-conv-kpt/plot_kpt_conv.py` converts each k-mesh into an
"inverse k-point distance" (a rough, mesh-independent measure of density) so
that runs with different `kptrlatt` can be compared on the same x-axis:

In [ ]:
k_recip_dist, ene_per_atom_eV = [], []

with abilab.GsrRobot.from_dir("flow_gaas_convkpt") as robot:
    for label, gsr in robot:
        ene_per_atom_eV.append(gsr.energy_per_atom)
        rprim = gsr.structure.lattice.matrix
        kptrlatt = gsr.kpoints.ksampling["kptrlatt"]
        R_latt = np.dot(kptrlatt, rprim)
        k_latt = 2 * np.pi * np.linalg.inv(R_latt)
        kmin = max(np.linalg.norm(k) for k in k_latt)
        k_recip_dist.append(1 / kmin)

ca = ConvergenceAnalyzer.from_xy_label_vals(
    "Inverse k-point distance (Ang)", k_recip_dist,
    "E/natom (eV)", ene_per_atom_eV, tols=1e-3)
ca.plot();

> **Exercise.** Redo the ecut convergence study for silicon
> (`wlib.si_structure()`), reusing `gs_input` as a template: build a flow
> with one SCF task per `ecut` (as in Part 1, Section 4.1), run it, then
> analyze it here with a `GsrRobot` and `ConvergenceAnalyzer` as above. Is
> the converged `ecut` similar to the one you found for GaAs? Should it be?

### 5.3 Equation of state and the lattice parameter

Results from the flow built in Part 1, Section 6 (`flow_gaas_eos/`): one
SCF task per isotropically-scaled volume of GaAs around the experimental
structure. A `GsrRobot` collects them, and `abilab.EOS` fits a
Birch-Murnaghan equation of state to $E(V)$ -- the same strategy used in
the AbiPy `base3` (silicon) lesson to determine the lattice parameter.


In [ ]:
with abilab.GsrRobot.from_dir("flow_gaas_eos") as robot:
    table = robot.get_dataframe()

table = table.sort_values("volume")
table[["volume", "energy"]]

In [ ]:
eos = abilab.EOS.Birch_Murnaghan()
fit = eos.fit(table["volume"].values, table["energy"].values)
print(fit)
fit.plot();

The fit reports the equilibrium volume $V_0$, bulk modulus $B_0$ and its
pressure derivative $B_0'$. From $V_0$ you can back out the equilibrium
lattice parameter $a_0 = (4 V_0)^{1/3}$ for this zinc-blende (4 atoms/cell)
structure, and compare it with the experimental value (~5.65 Ang for GaAs).

> **Exercise.** Compare the fitted `ecut=12` Ha equilibrium volume with
> what you'd get at `ecut=24` Ha. How much does $a_0$ shift? Is `ecut=12`
> Ha good enough for this particular quantity, even if it wasn't fully
> converged for the total energy in Section 4?

### 5.4 abirun.py, abiopen.py, abicomp.py

Every calculation in these two notebooks was expressed as a `Flow` made of
`Work`s made of `Task`s -- the same abstraction whether you're running one
SCF task or the hundreds of DFPT tasks behind the phonon flow. A few
commands worth knowing for automating and inspecting flows outside the
notebook:

```
abirun.py FLOWDIR scheduler      # run (or resume) a flow until completion
abirun.py FLOWDIR status         # check the status of all tasks
abiopen.py FILE                  # open any Abinit netcdf/output file in ipython
abiopen.py FILE -nb              # ... or generate a Jupyter notebook from it
abiopen.py FILE -e               # ... or get a quick summary plot ("expose")
abicomp.py gsr FILE1 FILE2 ...   # compare several files of the same kind -- the CLI equivalent of the Robots used above
```


## Wrap-up

Across these two notebooks we went from a bare `Structure` to: a converged
total energy, ecut/k-point convergence studies, band structures for two
materials, an equation-of-state fit for the lattice parameter, and a DFPT
phonon calculation with Born effective charges -- all driven by the same
handful of AbiPy abstractions (`AbinitInput`, `Flow`, `abiopen`, `Robot`).

For more worked examples, see the
[AbiPy Jupyter Book](https://github.com/abinit/abipy_book), the
[AbiPy plot gallery](https://abinit.github.io/abipy/gallery/index.html) and
the [AbiPy flow gallery](https://abinit.github.io/abipy/flow_gallery/index.html).
